In [ ]:
import json
import requests
import re
import sys
from pathlib import Path

# ================= CẤU HÌNH =================
INPUT_FILE = "HNMU_Chunks_Segment.json"
OUTPUT_FILE = "dataset_qa.json"
API_URL = "http://localhost:8080/v1/chat/completions"
TEMPERATURE = 0.1 
# ============================================

category = [        
    "Đăng ký học tập – tín chỉ – học kỳ",
    "Điều kiện học lại – cải thiện điểm – thi lại",
    "Đánh giá, chấm điểm, xếp loại học lực",
    "Cảnh báo học tập – buộc thôi học",
    "Đồ án, khóa luận, học phần thay thế",
    "Nghỉ học tạm thời – bảo lưu – thôi học",
    "Công nhận tín chỉ – miễn học – chứng chỉ ngoại ngữ & tin học",
    "Chuyển ngành, chuyển trường, học song ngành",
    "Điều kiện tốt nghiệp & xếp hạng bằng"
]

def construct_full_context(item):
    """
    LOGIC MỚI: Gộp các trường rời rạc (Level 1, Level 2, Content...) 
    thành một văn bản liền mạch để đưa vào Prompt.
    """
    parts = []
    
    # 1. Thêm Tiêu đề chương (Level 1)
    if item.get("Level 1"):
        parts.append(item["Level 1"].upper())
        
    # 2. Thêm Tiêu đề điều (Level 2)
    if item.get("Level 2"):
        parts.append(item["Level 2"])
        
    # 3. Thêm nội dung chính (Article)
    if item.get("Article"):
        parts.append(item["Article"])
        
    # 4. Thêm các mục chi tiết (Content - xử lý cả List hoặc String)
    content_list = item.get("Content", [])
    if isinstance(content_list, list):
        for line in content_list:
            parts.append(line)
    elif isinstance(content_list, str):
        parts.append(content_list)
        
    # Gộp lại bằng dấu xuống dòng
    return "\n".join(parts)

def create_prompt(context_chunk):
    """
    PROMPT CỦA BẠN (GIỮ NGUYÊN)
    """
    return f"""
Bạn là chuyên gia học vụ đại học và trợ lý pháp lý giáo dục.

NHIỆM VỤ:
- Đọc và hiểu toàn bộ nội dung quy chế dưới đây:
"{context_chunk}"

- Sinh DUY NHẤT 01 cặp CÂU HỎI – CÂU TRẢ LỜI (Q&A).
- Nội dung Q&A phải dựa DUY NHẤT trên văn bản trong tệp.
- TUYỆT ĐỐI KHÔNG suy diễn, KHÔNG dùng kiến thức bên ngoài, KHÔNG trả lời chung chung.

# PHẠM VI:
# - CHỈ sinh câu hỏi và câu trả lời cho CATEGORY sau:
#     {category}

YÊU CẦU VỀ CÂU HỎI:
- Mang tính THỰC TẾ, giống cách sinh viên thường hỏi trên:
    + Diễn đàn sinh viên
    + Nhóm Facebook
    + Phòng Quản lý đào tạo
- Có thể là:
    + Câu hỏi tình huống cụ thể
    + Câu hỏi dạng "em có được không?"
    + Câu hỏi dạng "nếu … thì sao?"
    + Một nhầm lẫn phổ biến của sinh viên
- Không hỏi theo kiểu máy móc, không nêu trực tiếp "Điều X quy định gì?"
- Không trùng ý với các câu hỏi phổ biến quá chung chung.

YÊU CẦU VỀ CÂU TRẢ LỜI:
- Trả lời ĐÚNG theo quy chế
- Diễn đạt rõ ràng, dễ hiểu, đúng ngôn ngữ học vụ
- Có thể diễn giải lại nhưng KHÔNG làm sai ý văn bản gốc
- Nếu cần, có thể nêu căn cứ Điều/Khoản trong quy chế (không bắt buộc)

ĐỊNH DẠNG OUTPUT (BẮT BUỘC – KHÔNG SINH THÊM NỘI DUNG KHÁC):

Q: <Câu hỏi>
A: <Câu trả lời>

NGÔN NGỮ:
- Tiếng Việt
- Văn phong sinh viên – học vụ – rõ ràng – chính xác

MỤC TIÊU:
- Kết quả dùng trực tiếp cho:
    + Chatbot tư vấn quy chế đào tạo
    + Fine-tune mô hình ngôn ngữ
    + RAG hỏi đáp học vụ
"""

def parse_qa_text(text, current_topic=""):
    """
    Phân tích văn bản thô (Q: ... A: ...) thành danh sách dictionary.
    Có cập nhật thêm tham số current_topic để điền tự động.
    """
    qa_list = []
    pattern = r"Q:\s*(.*?)\s*\nA:\s*(.*?)(?=\nQ:|$)"
    
    matches = re.findall(pattern, text, re.DOTALL | re.IGNORECASE)
    
    for q, a in matches:
        q_clean = q.strip()
        a_clean = a.strip()
        
        if q_clean and a_clean:
            qa_list.append({
                "Câu hỏi": q_clean,
                "Câu trả lời": a_clean,
                "Chủ đề": current_topic  # Tự động điền chủ đề lấy từ Level 2
            })
            
    return qa_list

def process_chunks():
    if not Path(INPUT_FILE).exists():
        print(f"❌ Không tìm thấy file: {INPUT_FILE}")
        return

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        chunks = json.load(f)

    print(f"📂 Đã tải {len(chunks)} chunks.")
    
    all_qa_pairs = []

    print("🚀 Bắt đầu xử lý...")

    for index, chunk in enumerate(chunks):
        # --- LOGIC MỚI: Xây dựng ngữ cảnh từ JSON cấu trúc ---
        if index > 5:
            break
        text_content = construct_full_context(chunk)
        
        # Lấy chủ đề từ Level 2 (Ví dụ: "Điều 2. Chương trình đào tạo...")
        # Nếu không có Level 2 thì để trống hoặc lấy Level 1
        topic = chunk.get("Level 2", chunk.get("Level 1", ""))
            
        if not text_content or len(str(text_content)) < 10:
            print(f"⚠️  Chunk {index}: Quá ngắn -> Bỏ qua.")
            continue

        print(f"\nProcessing chunk {index + 1}/{len(chunks)}...")

        payload = {
            "messages": [
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": create_prompt(text_content)}
            ],
            "temperature": TEMPERATURE,
            "max_tokens": 2048,
        }

        try:
            resp = requests.post(API_URL, json=payload)
            if resp.status_code == 200:
                raw_content = resp.json()['choices'][0]['message']['content']
                
                print(f"=== DATA ===\n{text_content}...\n")
                print(f"=== RAW ===\n{raw_content}\n")
                
                # Truyền topic vào hàm parse
                parsed_items = parse_qa_text(raw_content, current_topic=topic)
                
                if parsed_items:
                    print(f"   ✅ Trích xuất được {len(parsed_items)} cặp Q&A.")
                    all_qa_pairs.extend(parsed_items)
                    
                    # Lưu ngay lập tức
                    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
                        json.dump(all_qa_pairs, f, ensure_ascii=False, indent=4)
                else:
                    print(f"   ⚠️  Không tìm thấy định dạng Q:/A: trong phản hồi.")
            else:
                print(f"   ❌ Lỗi API: {resp.status_code}")

        except Exception as e:
            print(f"   ❌ Lỗi: {e}")

    print(f"\n🎉 HOÀN TẤT! Tổng số cặp câu hỏi: {len(all_qa_pairs)}")

if __name__ == "__main__":
    process_chunks()